<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DataWrangling/blob/main/W03_scaler_vergleich_ames_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚖️ Feature-Skalierung im Vergleich — Ames Housing Datensatz

**Data Wrangling & Feature Engineering — Praktisches Notebook**

Dieses Notebook vergleicht die wichtigsten Skalierungsverfahren aus scikit-learn  
anhand des **Ames Housing Datensatzes** (Hauspreise in Ames, Iowa).

---

## Welche Scaler werden verglichen?

| Scaler | Transformation | Wertebereich | Besonders geeignet für |
|--------|---------------|--------------|------------------------|
| `StandardScaler` | z = (x − μ) / σ | unbegrenzt (~−3 bis +3) | Normalverteilte Features, lineare Modelle |
| `MinMaxScaler` | z = (x − min) / (max − min) | [0, 1] | Bildpixel, neuronale Netze |
| `RobustScaler` | z = (x − Median) / IQR | unbegrenzt | Datensätze mit vielen Ausreissern |
| `MaxAbsScaler` | z = x / max(abs(x)) | [−1, 1] | Sparse-Daten |
| `Normalizer` | Normiert jede **Zeile** auf Länge 1 | [−1, 1] | Textdaten, Kosinusähnlichkeit |
| `PowerTransformer` | Box-Cox / Yeo-Johnson | ~Normal | Schiefe Verteilungen |
| `QuantileTransformer` | Mappt auf Normalverteilung | ~N(0,1) | Outlier, nichtlineare Verteilungen |

---

**Aufbau:**
- **Teil 1** — Datensatz laden & verstehen
- **Teil 2** — Vorher/Nachher-Plots für jeden Scaler
- **Teil 3** — Auswirkung auf ML-Performance (5 Modelle × 7 Scaler)
- **Teil 4** — Zusammenfassung & Empfehlungen

---
## 🔧 Setup — Pakete installieren und importieren

In [ ]:
# Alle benötigten Pakete sind in Google Colab vorinstalliert
# Nur scikit-learn-datasets für Ames Housing (ab sklearn 1.0 verfügbar)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "scikit-learn>=1.0", "pandas", "matplotlib", "seaborn", "numpy"],
               capture_output=True)
print("✅ Pakete bereit.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error

# Modelle
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Scaler
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    MaxAbsScaler, Normalizer, PowerTransformer, QuantileTransformer
)

# Reproduzierbarkeit
np.random.seed(42)

# Plot-Stil
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

print("✅ Imports abgeschlossen.")

---
# Teil 1 — Datensatz laden & verstehen

Der **Ames Housing Datensatz** enthält Verkaufspreise von 1460 Häusern  
in Ames, Iowa (2006–2010) mit 79 erklärenden Merkmalen.

**Zielvariable:** `SalePrice` (Verkaufspreis in US-Dollar)  
**Verwendete Features:** Nur numerische Spalten (für Skalierungsdemonstration)

In [ ]:
# ── Datensatz laden ──────────────────────────────────────────────────────────
print("Lade Ames Housing Datensatz von OpenML...")
ames_raw = fetch_openml(name="house_prices", as_frame=True, parser="auto")
df_full = ames_raw.frame.copy()

print(f"Datensatz geladen: {df_full.shape[0]} Zeilen × {df_full.shape[1]} Spalten")

# Zielvariable und numerische Features trennen
target_col = "SalePrice"
df_full[target_col] = pd.to_numeric(df_full[target_col], errors="coerce")

# Alle numerischen Spalten extrahieren (außer Zielvariable)
num_cols_all = df_full.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols_all if c != target_col]

# Fehlende Werte mit Median auffüllen (für saubere Skalierungsdemonstration)
df = df_full[num_cols + [target_col]].copy()
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
median_vals = df[num_cols].median()
df[num_cols] = df[num_cols].fillna(median_vals)

X = df[num_cols].values
y = df[target_col].values

print(f"\nNumerische Features: {len(num_cols)}")
print(f"Zielvariable: {target_col}")
print(f"Preisspanne: ${y.min():,.0f} – ${y.max():,.0f}")
print(f"Durchschnittspreis: ${y.mean():,.0f}")

In [ ]:
# ── Überblick über die Daten ─────────────────────────────────────────────────
print("📋 Erste 5 Zeilen (Auswahl der wichtigsten Features):")
wichtige_features = ["LotArea", "GrLivArea", "OverallQual", "YearBuilt",
                     "TotalBsmtSF", "GarageArea", "SalePrice"]
display(df[wichtige_features].head())

print("\n📊 Statistische Kennzahlen:")
display(df[wichtige_features].describe().round(1))

In [ ]:
# ── Verteilung der Zielvariable ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Originalverteilung
axes[0].hist(y, bins=50, color="#4878cf", edgecolor="white", alpha=0.85)
axes[0].axvline(np.mean(y), color="red", ls="--", lw=1.8, label=f"Mittelwert: ${np.mean(y):,.0f}")
axes[0].axvline(np.median(y), color="orange", ls="--", lw=1.8, label=f"Median: ${np.median(y):,.0f}")
axes[0].set_title("SalePrice — Originalverteilung", fontweight="bold")
axes[0].set_xlabel("Preis (USD)")
axes[0].set_ylabel("Häufigkeit")
axes[0].legend()

# Log-Verteilung (zeigt Normalverteilungsnähe)
axes[1].hist(np.log1p(y), bins=50, color="#6acc65", edgecolor="white", alpha=0.85)
axes[1].set_title("log(SalePrice) — Log-Transformation", fontweight="bold")
axes[1].set_xlabel("log(1 + Preis)")
axes[1].set_ylabel("Häufigkeit")

plt.suptitle("Verteilung der Zielvariable", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("💡 Beobachtung: SalePrice ist rechtschief (right-skewed).")
print("   Die Log-Transformation ergibt eine annähernd normalverteilte Zielvariable.")

In [ ]:
# ── Verteilung ausgewählter Features (vor Skalierung) ────────────────────────
demo_features = ["LotArea", "GrLivArea", "OverallQual", "YearBuilt",
                 "TotalBsmtSF", "GarageArea"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(demo_features):
    vals = df[feat].values
    q25, q75 = np.percentile(vals, [25, 75])
    iqr = q75 - q25
    n_outlier = np.sum((vals < q25 - 1.5*iqr) | (vals > q75 + 1.5*iqr))

    axes[i].hist(vals, bins=40, color="#4878cf", edgecolor="white", alpha=0.8)
    axes[i].set_title(f"{feat}", fontweight="bold")
    axes[i].set_xlabel("Wert")
    axes[i].set_ylabel("Häufigkeit")
    axes[i].text(0.97, 0.95, f"Ausreisser: {n_outlier}",
                 transform=axes[i].transAxes, ha="right", va="top",
                 fontsize=9, color="firebrick",
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))

plt.suptitle("Feature-Verteilungen VOR der Skalierung", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Teil 2 — Vorher/Nachher-Plots für jeden Scaler

Für jeden Scaler werden **zwei repräsentative Features** verglichen:
- `GrLivArea` (Wohnfläche über Erdgeschoss) — rechtsschiefe Verteilung mit Ausreissern
- `OverallQual` (Gesamtqualität, 1–10) — diskrete, annähernd symmetrische Verteilung

In [ ]:
# ── Scaler-Konfiguration ────────────────────────────────────────────────────
SCALER_DICT = {
    "Kein Scaler\n(Original)": None,
    "StandardScaler\nz = (x−μ)/σ": StandardScaler(),
    "MinMaxScaler\n[0, 1]": MinMaxScaler(),
    "RobustScaler\n(Median/IQR)": RobustScaler(),
    "MaxAbsScaler\n[−1, 1]": MaxAbsScaler(),
    "PowerTransformer\n(Yeo-Johnson)": PowerTransformer(method="yeo-johnson"),
    "QuantileTransformer\n→ Normalverteilung": QuantileTransformer(
        output_distribution="normal", random_state=42),
}

# Demo-Features für Vorher/Nachher
demo_feat_indices = {
    "GrLivArea (Wohnfläche, schief + Ausreisser)": num_cols.index("GrLivArea"),
    "OverallQual (Qualität, diskret 1–10)": num_cols.index("OverallQual"),
}

print("✅ Scaler-Konfiguration geladen.")
print("\nFolgende Scaler werden verglichen:")
for name in SCALER_DICT:
    print(f"  • {name.split(chr(10))[0]}")

In [ ]:
# ── Vorher/Nachher-Plot: GrLivArea ──────────────────────────────────────────
feat_name = "GrLivArea"
feat_idx = num_cols.index(feat_name)
raw_vals = X[:, feat_idx].reshape(-1, 1)

n_scalers = len(SCALER_DICT)
fig, axes = plt.subplots(2, n_scalers, figsize=(20, 7), sharey="row")

colors_before = "#4878cf"
colors_after  = "#6acc65"

for col_i, (scaler_label, scaler) in enumerate(SCALER_DICT.items()):
    # Originalwerte oben
    ax_top = axes[0, col_i]
    ax_top.hist(raw_vals, bins=40, color=colors_before, edgecolor="white", alpha=0.85)
    ax_top.set_title(scaler_label, fontsize=8.5, fontweight="bold")
    if col_i == 0:
        ax_top.set_ylabel("Vorher", fontsize=10, fontweight="bold", color=colors_before)
    ax_top.tick_params(axis="x", labelsize=7)
    ax_top.tick_params(axis="y", labelsize=7)

    # Skalierte Werte unten
    ax_bot = axes[1, col_i]
    if scaler is None:
        transformed = raw_vals
        ax_bot.set_xlabel("Originalwert", fontsize=7)
    else:
        transformed = scaler.fit_transform(raw_vals)
        ax_bot.set_xlabel("Skalierter Wert", fontsize=7)

    ax_bot.hist(transformed, bins=40, color=colors_after, edgecolor="white", alpha=0.85)
    if col_i == 0:
        ax_bot.set_ylabel("Nachher", fontsize=10, fontweight="bold", color=colors_after)
    ax_bot.tick_params(axis="x", labelsize=7)
    ax_bot.tick_params(axis="y", labelsize=7)

    # Statistik einblenden
    mu, sd = transformed.mean(), transformed.std()
    mn, mx = transformed.min(), transformed.max()
    ax_bot.text(0.97, 0.95,
                f"μ={mu:.2f}\nσ={sd:.2f}\n[{mn:.1f}, {mx:.1f}]",
                transform=ax_bot.transAxes, ha="right", va="top",
                fontsize=7, family="monospace",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="lightyellow", alpha=0.85))

fig.suptitle(f"Scaler-Vergleich — Feature: {feat_name} (Wohnfläche über EG in ft²)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Vorher/Nachher-Plot: OverallQual ────────────────────────────────────────
feat_name = "OverallQual"
feat_idx = num_cols.index(feat_name)
raw_vals = X[:, feat_idx].reshape(-1, 1)

fig, axes = plt.subplots(2, n_scalers, figsize=(20, 7), sharey="row")

for col_i, (scaler_label, scaler) in enumerate(SCALER_DICT.items()):
    ax_top = axes[0, col_i]
    ax_top.hist(raw_vals, bins=10, color=colors_before, edgecolor="white", alpha=0.85)
    ax_top.set_title(scaler_label, fontsize=8.5, fontweight="bold")
    if col_i == 0:
        ax_top.set_ylabel("Vorher", fontsize=10, fontweight="bold", color=colors_before)
    ax_top.tick_params(axis="x", labelsize=7)
    ax_top.tick_params(axis="y", labelsize=7)

    ax_bot = axes[1, col_i]
    if scaler is None:
        transformed = raw_vals
    else:
        transformed = scaler.fit_transform(raw_vals)

    ax_bot.hist(transformed, bins=10, color=colors_after, edgecolor="white", alpha=0.85)
    if col_i == 0:
        ax_bot.set_ylabel("Nachher", fontsize=10, fontweight="bold", color=colors_after)
    ax_bot.tick_params(axis="x", labelsize=7)
    ax_bot.tick_params(axis="y", labelsize=7)

    mu, sd = transformed.mean(), transformed.std()
    mn, mx = transformed.min(), transformed.max()
    ax_bot.text(0.97, 0.95,
                f"μ={mu:.2f}\nσ={sd:.2f}\n[{mn:.1f}, {mx:.1f}]",
                transform=ax_bot.transAxes, ha="right", va="top",
                fontsize=7, family="monospace",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="lightyellow", alpha=0.85))

fig.suptitle(f"Scaler-Vergleich — Feature: {feat_name} (Gesamtqualität, diskret 1–10)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplots: Skalierte Wertebereiche im Überblick ──────────────────────────
# Zeigt für alle Scaler den Wertebereich von GrLivArea als Boxplot nebeneinander

feat_name = "GrLivArea"
feat_idx  = num_cols.index(feat_name)
raw_vals  = X[:, feat_idx].reshape(-1, 1)

scaled_data = {}
for label, scaler in SCALER_DICT.items():
    short_label = label.split("\n")[0]
    if scaler is None:
        scaled_data[short_label] = raw_vals.flatten()
    else:
        scaled_data[short_label] = scaler.fit_transform(raw_vals).flatten()

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(list(scaled_data.values()),
                labels=list(scaled_data.keys()),
                patch_artist=True,
                medianprops=dict(color="black", linewidth=2))

palette = sns.color_palette("muted", len(scaled_data))
for patch, color in zip(bp["boxes"], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_title(f"Wertebereich nach Skalierung — {feat_name}", fontweight="bold", fontsize=12)
ax.set_ylabel("Skalierter Wert")
ax.set_xlabel("Skalierungsverfahren")
ax.tick_params(axis="x", labelsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Beobachtung:")
print("  • StandardScaler / RobustScaler: unbegrenzter Wertebereich, unterschiedliche Ausreisserbehandlung")
print("  • MinMaxScaler / MaxAbsScaler:   fester Wertebereich, empfindlich gegen Ausreisser")
print("  • PowerTransformer / Quantile:   transformiert die Verteilungsform selbst")

In [ ]:
# ── KDE-Plots: Alle Scaler im Vergleich ─────────────────────────────────────
# Kernel Density Estimation zeigt Verteilungsform klarer als Histogramm

feat_name = "GrLivArea"
feat_idx  = num_cols.index(feat_name)
raw_vals  = X[:, feat_idx].reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gruppe 1: Position-unabhängige Scaler
group1 = {
    "Original": None,
    "StandardScaler": StandardScaler(),
    "RobustScaler": RobustScaler(),
}
# Gruppe 2: Bereichsbeschränkte Scaler
group2 = {
    "Original": None,
    "MinMaxScaler": MinMaxScaler(),
    "MaxAbsScaler": MaxAbsScaler(),
}
# Gruppe 3: Transformationsbasierte Scaler
group3 = {
    "Original (norm.)": None,
    "PowerTransformer": PowerTransformer(method="yeo-johnson"),
    "QuantileTransformer": QuantileTransformer(output_distribution="normal", random_state=42),
}

for ax, group, title in zip(axes,
                             [group1, group2, group3],
                             ["Standardisierung", "Normierung", "Verteilungstransformation"]):
    for label, scaler in group.items():
        if scaler is None:
            vals = raw_vals.flatten()
            # Auf [0,1] normieren für Vergleichbarkeit in Gruppe 1 & 2
            vals_norm = (vals - vals.min()) / (vals.max() - vals.min())
            sns.kdeplot(vals_norm, ax=ax, label=label, lw=2, ls="--", color="gray")
        else:
            vals = scaler.fit_transform(raw_vals).flatten()
            # Für KDE: auf [0,1] normieren damit Kurven vergleichbar sind
            v_min, v_max = vals.min(), vals.max()
            if v_max > v_min:
                vals = (vals - v_min) / (v_max - v_min)
            sns.kdeplot(vals, ax=ax, label=label, lw=2)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Normierter Wert")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle(f"KDE-Vergleich: Verteilungsform nach Skalierung — {feat_name}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Teil 3 — Auswirkung der Skalierung auf ML-Performance

**Fragestellung:** Welcher Scaler führt bei welchem Modell zur besten Vorhersagequalität?

**Versuchsaufbau:**
- 5-fache Kreuzvalidierung (`KFold, k=5, shuffle=True`)
- Metriken: R² (Bestimmtheitsmaß) und RMSE (Wurzel des mittleren quadratischen Fehlers)
- Modelle:
  - `Ridge` (lineare Regression mit L2-Regularisierung) — sehr skalierungsabhängig
  - `Lasso` (lineare Regression mit L1-Regularisierung) — sehr skalierungsabhängig
  - `KNeighborsRegressor` (k-Nearest Neighbours) — sehr skalierungsabhängig
  - `SVR` (Support Vector Regression) — sehr skalierungsabhängig
  - `RandomForest` (Entscheidungsbäume) — skalierungsunabhängig als Referenz

> ⚠️ **Wichtig:** Skalierung immer **nur auf Trainingsdaten** fitten (`.fit_transform`),  
> dann auf Testdaten **anwenden** (`.transform`). Kein Data Leakage!

In [ ]:
# ── Modell- und Scaler-Konfiguration für Benchmarking ──────────────────────
MODELLE = {
    "Ridge":          Ridge(alpha=1.0),
    "Lasso":          Lasso(alpha=10.0, max_iter=5000),
    "KNN (k=5)":      KNeighborsRegressor(n_neighbors=5),
    "SVR":            SVR(kernel="rbf", C=100, gamma="scale"),
    "RandomForest":   RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

# Scaler ohne Normalizer (Normalizer normiert Zeilen, nicht Spalten — andere Semantik)
SCALER_BENCH = {
    "Kein Scaler":         None,
    "StandardScaler":      StandardScaler(),
    "MinMaxScaler":        MinMaxScaler(),
    "RobustScaler":        RobustScaler(),
    "MaxAbsScaler":        MaxAbsScaler(),
    "PowerTransformer":    PowerTransformer(method="yeo-johnson"),
    "QuantileTransformer": QuantileTransformer(output_distribution="normal", random_state=42),
}

print("✅ Konfiguration geladen.")
print(f"\n{len(MODELLE)} Modelle × {len(SCALER_BENCH)} Scaler = {len(MODELLE)*len(SCALER_BENCH)} Experimente")

In [ ]:
# ── Kreuzvalidierung durchführen ────────────────────────────────────────────
# Hinweis: Der Scaler wird innerhalb der Pipeline fit_transform(X_train)
# und transform(X_test) aufgerufen — kein Data Leakage!

from sklearn.pipeline import make_pipeline
import time

cv = KFold(n_splits=5, shuffle=True, random_state=42)

ergebnisse = []  # Liste aller Ergebnisse

print("Starte Kreuzvalidierung...")
print("-" * 55)

start_total = time.time()

for modell_name, modell in MODELLE.items():
    for scaler_name, scaler in SCALER_BENCH.items():
        # Pipeline erstellt: Scaler → Modell (sicherer Ablauf ohne Datenleck)
        if scaler is not None:
            pipeline = make_pipeline(scaler, modell)
        else:
            pipeline = modell  # Kein Scaler

        # R²-Score (5-fache CV)
        r2_scores = cross_val_score(
            pipeline, X, y,
            cv=cv, scoring="r2", n_jobs=-1
        )
        # Negativer RMSE (sklearn maximiert, daher negativ)
        neg_rmse = cross_val_score(
            pipeline, X, y,
            cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
        )

        ergebnisse.append({
            "Modell":   modell_name,
            "Scaler":   scaler_name,
            "R²_mean":  r2_scores.mean(),
            "R²_std":   r2_scores.std(),
            "RMSE_mean": (-neg_rmse).mean(),
            "RMSE_std":  (-neg_rmse).std(),
        })

    print(f"  ✓ {modell_name} fertig")

print("-" * 55)
print(f"Gesamtzeit: {time.time()-start_total:.1f}s")
print("\n✅ Kreuzvalidierung abgeschlossen.")

df_results = pd.DataFrame(ergebnisse)

In [ ]:
# ── Ergebnistabelle anzeigen ────────────────────────────────────────────────
print("\n📊 Ergebnisse (R² und RMSE, 5-fache Kreuzvalidierung):")
print("=" * 70)

# Beste Scaler pro Modell hervorheben
for modell_name in MODELLE.keys():
    subset = df_results[df_results["Modell"] == modell_name].copy()
    subset = subset.sort_values("R²_mean", ascending=False)
    print(f"\n🔷 {modell_name}:")
    subset_display = subset[["Scaler", "R²_mean", "R²_std", "RMSE_mean"]].copy()
    subset_display["R²_mean"] = subset_display["R²_mean"].map("{:.4f}".format)
    subset_display["R²_std"] = subset_display["R²_std"].map("±{:.4f}".format)
    subset_display["RMSE_mean"] = subset_display["RMSE_mean"].map("{:,.0f} $".format)
    display(subset_display.reset_index(drop=True))

In [ ]:
# ── Plot 1: R²-Heatmap (Modelle × Scaler) ───────────────────────────────────
pivot_r2 = df_results.pivot(index="Modell", columns="Scaler", values="R²_mean")

# Spaltenreihenfolge festlegen
scaler_order = list(SCALER_BENCH.keys())
pivot_r2 = pivot_r2[scaler_order]

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(pivot_r2.values, cmap="RdYlGn", aspect="auto", vmin=0.5, vmax=1.0)

# Achsenbeschriftungen
ax.set_xticks(range(len(pivot_r2.columns)))
ax.set_xticklabels(pivot_r2.columns, rotation=30, ha="right", fontsize=10)
ax.set_yticks(range(len(pivot_r2.index)))
ax.set_yticklabels(pivot_r2.index, fontsize=10)

# Werte in Zellen
for i in range(len(pivot_r2.index)):
    for j in range(len(pivot_r2.columns)):
        val = pivot_r2.values[i, j]
        text_color = "black" if 0.6 < val < 0.9 else "white"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=9, color=text_color, fontweight="bold")

plt.colorbar(im, ax=ax, label="R² (höher = besser)")
ax.set_title("R²-Score je Modell und Scaler\n(5-fache Kreuzvalidierung, Ames Housing)",
             fontsize=12, fontweight="bold", pad=15)
ax.set_xlabel("Skalierungsverfahren", fontsize=11)
ax.set_ylabel("Modell", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Grouped Bar Chart — R² je Scaler, gruppiert nach Modell ─────────
fig, ax = plt.subplots(figsize=(16, 6))

modell_names = list(MODELLE.keys())
scaler_names = list(SCALER_BENCH.keys())
n_modelle  = len(modell_names)
n_scaler   = len(scaler_names)
bar_width  = 0.12
x_pos      = np.arange(n_modelle)

palette = sns.color_palette("tab10", n_scaler)

for s_i, scaler_name in enumerate(scaler_names):
    r2_vals = [
        df_results[(df_results["Modell"] == m) & (df_results["Scaler"] == scaler_name)]["R²_mean"].values[0]
        for m in modell_names
    ]
    r2_stds = [
        df_results[(df_results["Modell"] == m) & (df_results["Scaler"] == scaler_name)]["R²_std"].values[0]
        for m in modell_names
    ]
    offset = (s_i - n_scaler / 2 + 0.5) * bar_width
    bars = ax.bar(x_pos + offset, r2_vals, bar_width,
                  label=scaler_name, color=palette[s_i], alpha=0.85,
                  yerr=r2_stds, capsize=2, error_kw=dict(elinewidth=0.8))

ax.set_xticks(x_pos)
ax.set_xticklabels(modell_names, fontsize=11)
ax.set_ylabel("R²-Score (Kreuzvalidierung)", fontsize=11)
ax.set_title("Einfluss der Skalierung auf R²-Score je Modell",
             fontsize=12, fontweight="bold")
ax.legend(title="Scaler", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.set_ylim(bottom=max(0, df_results["R²_mean"].min() - 0.1))
ax.grid(axis="y", alpha=0.3)
ax.set_xlabel("Modell", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: RMSE-Vergleich ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(MODELLE), figsize=(18, 5), sharey=False)

for ax, (modell_name, _) in zip(axes, MODELLE.items()):
    subset = df_results[df_results["Modell"] == modell_name].copy()
    subset = subset.sort_values("RMSE_mean")

    colors = [palette[list(SCALER_BENCH.keys()).index(s)] for s in subset["Scaler"]]
    bars = ax.barh(subset["Scaler"], subset["RMSE_mean"],
                   xerr=subset["RMSE_std"], capsize=3,
                   color=colors, alpha=0.85, error_kw=dict(elinewidth=0.8))

    ax.set_title(modell_name, fontweight="bold", fontsize=10)
    ax.set_xlabel("RMSE (USD)", fontsize=9)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="x", alpha=0.3)

    # Besten Scaler markieren
    best_idx = subset["RMSE_mean"].idxmin()
    best_scaler = subset.loc[best_idx, "Scaler"]
    for bar, scaler_name_bar in zip(bars, subset["Scaler"]):
        if scaler_name_bar == best_scaler:
            bar.set_edgecolor("black")
            bar.set_linewidth(2.0)

plt.suptitle("RMSE je Modell und Scaler (niedriger = besser)\nBester Scaler je Modell = schwarzer Rahmen",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Relative Verbesserung durch Skalierung ───────────────────────────
# Wie viel bringt der beste Scaler gegenüber "Kein Scaler"?

fig, ax = plt.subplots(figsize=(10, 5))

verbesserungen = []
for modell_name in MODELLE.keys():
    subset = df_results[df_results["Modell"] == modell_name]
    r2_kein_scaler = subset[subset["Scaler"] == "Kein Scaler"]["R²_mean"].values[0]
    r2_bester       = subset["R²_mean"].max()
    bester_scaler   = subset.loc[subset["R²_mean"].idxmax(), "Scaler"]
    delta = r2_bester - r2_kein_scaler
    verbesserungen.append({
        "Modell": modell_name,
        "Δ R²":   delta,
        "Bester Scaler": bester_scaler,
    })

df_verbesserung = pd.DataFrame(verbesserungen).sort_values("Δ R²", ascending=True)

farben = ["#d73027" if v < 0 else "#1a9850" for v in df_verbesserung["Δ R²"]]
bars = ax.barh(df_verbesserung["Modell"], df_verbesserung["Δ R²"],
               color=farben, alpha=0.85)

# Besten Scaler als Label
for bar, (_, row) in zip(bars, df_verbesserung.iterrows()):
    x_pos_label = bar.get_width() + (0.002 if bar.get_width() >= 0 else -0.002)
    ha = "left" if bar.get_width() >= 0 else "right"
    ax.text(x_pos_label, bar.get_y() + bar.get_height()/2,
            f" {row['Bester Scaler']}", va="center", ha=ha, fontsize=9)

ax.axvline(0, color="black", lw=1.2)
ax.set_xlabel("Δ R² (Bester Scaler minus Kein Scaler)", fontsize=11)
ax.set_title("Relative Verbesserung durch den besten Scaler\n(grün = Skalierung hilft, rot = Skalierung schadet)",
             fontsize=11, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 5: Skalierungssensitivität je Feature ──────────────────────────────
# Wie stark ändert sich die R²-Varianz beim Ridge-Modell über alle Scaler?
# Features mit hoher Varianz profitieren am meisten von der richtigen Skalierung.

# Nur Ridge als skalierungssensitivstes Modell betrachten
ridge_results = df_results[df_results["Modell"] == "Ridge"].copy()

fig, ax = plt.subplots(figsize=(11, 4))

r2_vals = ridge_results["R²_mean"].values
r2_stds = ridge_results["R²_std"].values
scaler_labels = ridge_results["Scaler"].values

farben_bar = [palette[list(SCALER_BENCH.keys()).index(s)] for s in scaler_labels]
x = np.arange(len(scaler_labels))
bars = ax.bar(x, r2_vals, color=farben_bar, alpha=0.85,
              yerr=r2_stds, capsize=4, error_kw=dict(elinewidth=1.0))
ax.set_xticks(x)
ax.set_xticklabels(scaler_labels, rotation=25, ha="right", fontsize=10)
ax.set_ylabel("R²-Score", fontsize=11)
ax.set_title("Ridge Regression: R²-Score je Scaler\n(zeigt maximale Skalierungsabhängigkeit)",
             fontsize=11, fontweight="bold")
ax.set_ylim(bottom=max(0, r2_vals.min() - 0.1))
ax.grid(axis="y", alpha=0.3)

# Bester Scaler markieren
bester_idx = np.argmax(r2_vals)
bars[bester_idx].set_edgecolor("black")
bars[bester_idx].set_linewidth(2.5)
ax.annotate("Bester Scaler", xy=(bester_idx, r2_vals[bester_idx]),
            xytext=(bester_idx + 0.5, r2_vals[bester_idx] + 0.01),
            fontsize=9, arrowprops=dict(arrowstyle="->", color="black"))
plt.tight_layout()
plt.show()

---
# Teil 4 — Zusammenfassung & Empfehlungen

In [ ]:
# ── Zusammenfassungstabelle: Bester Scaler je Modell ────────────────────────
print("=" * 65)
print("  ERGEBNISZUSAMMENFASSUNG — BESTER SCALER JE MODELL")
print("=" * 65)

summary_rows = []
for modell_name in MODELLE.keys():
    subset = df_results[df_results["Modell"] == modell_name]
    best_row = subset.loc[subset["R²_mean"].idxmax()]
    worst_row = subset.loc[subset["R²_mean"].idxmin()]
    kein_scaler_r2 = subset[subset["Scaler"] == "Kein Scaler"]["R²_mean"].values[0]
    summary_rows.append({
        "Modell":           modell_name,
        "Bester Scaler":    best_row["Scaler"],
        "Bestes R²":        f"{best_row['R²_mean']:.4f}",
        "Schlechtester Scaler": worst_row["Scaler"],
        "Schlechtestes R²": f"{worst_row['R²_mean']:.4f}",
        "Δ R² (best−kein)": f"{best_row['R²_mean'] - kein_scaler_r2:+.4f}",
        "RMSE (best)":      f"${best_row['RMSE_mean']:,.0f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

## 📚 Lernziele — Was haben wir gelernt?

### 1. Skalierung ändert die Verteilungsform — je nach Verfahren unterschiedlich

| Verfahren | Was bleibt gleich? | Was ändert sich? |
|-----------|-------------------|------------------|
| `StandardScaler` | Verteilungsform | Mittelwert → 0, Std → 1 |
| `MinMaxScaler` | Verteilungsform | Wertebereich → [0,1] |
| `RobustScaler` | Verteilungsform | Median → 0, IQR → 1 (Ausreisser weniger Einfluss) |
| `MaxAbsScaler` | Verteilungsform + Nullpunkt | Wertebereich → [−1,1] |
| `PowerTransformer` | Wertebereich (ändert sich!) | **Verteilungsform** → Normalverteilung |
| `QuantileTransformer` | Wertebereich (ändert sich!) | **Verteilungsform** → exakte Normalverteilung |

---

### 2. Welche Modelle profitieren von Skalierung?

| Modell | Skalierungsabhängigkeit | Empfehlung |
|--------|------------------------|------------|
| `Ridge / Lasso / ElasticNet` | ⚠️ **Hoch** | StandardScaler oder RobustScaler |
| `KNN` | ⚠️ **Sehr hoch** | StandardScaler oder MinMaxScaler |
| `SVR` | ⚠️ **Sehr hoch** | StandardScaler |
| `Logistic / LinearRegression` | ⚠️ **Hoch** | StandardScaler |
| `Neuronale Netze` | ⚠️ **Hoch** | MinMaxScaler [0,1] oder StandardScaler |
| `RandomForest / XGBoost` | ✅ **Keine** | Skalierung hat keine Auswirkung |
| `Entscheidungsbäume` | ✅ **Keine** | Skalierung hat keine Auswirkung |

---

### 3. Praxisregel: Wann welchen Scaler wählen?

```
Gibt es viele Ausreisser?
    JA  → RobustScaler
    NEIN → Ist die Verteilung stark schief?
               JA  → PowerTransformer (Yeo-Johnson) oder QuantileTransformer
               NEIN → StandardScaler (Standardempfehlung)

Brauche ich Werte im Bereich [0, 1]? (z.B. Neuronale Netze, Bildpixel)
    → MinMaxScaler

Sind die Daten sparse (viele Nullen)?
    → MaxAbsScaler (erhält Nullen!)
```

---

### ⚠️ Wichtigste Regel: Kein Data Leakage!

```python
# RICHTIG ✅ — Scaler nur auf Trainingsdaten fitten
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled  = scaler.transform(X_test)        # nur transform!

# FALSCH ❌ — Scaler auf allen Daten fitten (Datenleck!)
scaler.fit_transform(X)  # Testdaten beeinflussen den Scaler!
```